# Process EMG recordings with bioread

Run one cell at a time. Choose the source files and raw ZM/CS channels, then use the revised `biopac_*` functions for FIR → comb → RMS, followed by `percent_mvc`.

Keep raw, clean RMS and %MVC arrays for each muscle. The export helpers save these six channels to MAT, NPZ and CSV, with extended metadata in JSON and NPZ. Retain the original `.acq`.

Exact AcqKnowledge equivalence and Windows import remain unverified. The export example is a template to use once processing is complete.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import bioread
import h5py
import numpy as np
import pandas as pd
from scipy import signal

### Root folder - Study id number, suffix of the files

Creating the path to read the acq files

In [ ]:
folder_root_path = r"I:/Research/Shares/dover_research/Div_fEMG/"

In [ ]:
study_id = "1011"
files_suffix = {
    "APP": Path(f"{folder_root_path}{study_id}_fEMG_div_2APP_raw.acq"),
    "PsychoPy": Path(f"{folder_root_path}{study_id}_fEMG_div_3PsychoPy_raw.acq"),
    "SlowRate": Path(f"{folder_root_path}{study_id}_fEMG_div_4SlowRate_raw.acq"),
}

### 3. Check input paths

All three rows must show `True` before loading. Correct the paths above if a file is missing.

In [ ]:
pd.DataFrame([
    {"task": task, "path": str(path.resolve()), "exists": path.is_file()}
    for task, path in files_suffix.items()
])

### 4. Read the ACQ files

Read each recording with `bioread`. Hidden channels are included automatically. Each loaded file prints a confirmation.

[bioread read API](https://github.com/uwmadison-chm/bioread/blob/main/bioread/__init__.py). `read()` accepts a Path; `read_file()` is its deprecated alias.

In [ ]:
acq_data = {}
for task, path in files_suffix.items():
    acq_data[task] = bioread.read(path)
    print(task, ":", len(acq_data[task].channels), "channels loaded")

### 5. List all channels, including hidden ones

Inspect the channel names, units, and sampling rates. Find the original raw channels 3 and 4; the Python `index` column may use different numbers.

[bioread channel-order note](https://github.com/uwmadison-chm/bioread#notes). Do not assume that list index 3 is AcqKnowledge channel 3.

In [ ]:
channel_rows = []
for task, recording in acq_data.items():
    for index, channel in enumerate(recording.channels):
        channel_rows.append({"task": task, "index": index, "order_num": channel.order_num,
                             "name": channel.name, "units": channel.units,
                             "rate_hz": channel.samples_per_second, "samples": len(channel.data)})
pd.DataFrame(channel_rows)

### 6. Choose the two raw channels

Replace each `None` with the **index** from the table above. Confirm which channel is ZM and which is CS. The next cell checks your choices.

In [ ]:
channel_indices = {
    "APP": {"ZM": None, "CS": None},
    "PsychoPy": {"ZM": None, "CS": None},
    "SlowRate": {"ZM": None, "CS": None},
}
pd.DataFrame(channel_indices).T

### 7. Select ZM and CS

Keep references to the two selected channels per recording. Other channels are excluded from processing; source files are not changed. Check the names printed below.

In [ ]:
selected = {}
for task, indices in channel_indices.items():
    for muscle, index in indices.items():
        assert type(index) is int, f"Choose the {task} {muscle} index"
        assert 0 <= index < len(acq_data[task].channels), f"Invalid index: {task} {muscle}"
        selected[task, muscle] = acq_data[task].channels[index]
        print(task, muscle, "->", selected[task, muscle].name)
    assert indices["ZM"] != indices["CS"], f"{task}: choose different indices for ZM and CS"

### Select data for ZM

We will create the steps for this channel first, if works we will apply that to allchannels and files

In [ ]:
zm_raw = selected["APP", "ZM"].data

In [ ]:
low_hz, high_hz = 28, 500
# Blackman -61 dB weights from the BIOPAC manual.
fir_window = [0.44959, 0.49364, 0.05677, 0.00000]
fir_scale = True
fir_padding = "reflect"

### FIR bandpass filter

In [ ]:
def fir_bandpass(data, sampling_rate):
    # TODO: check rounding via Channel Info > Modification History > FIR Filter.
    # BIOPAC 5 manual, p. 507, shows the reported Number of Coefficients there.
    num_taps = int(4 * sampling_rate / low_hz)

    coefficients = signal.firwin(
        num_taps, [low_hz, high_hz], fs=sampling_rate,
        pass_zero=False, window="boxcar", scale=False
    )
    # TODO: verify window sampling by comparing a centered impulse filtered in both programs.
    coefficients *= signal.windows.general_cosine(num_taps, fir_window)
    if fir_scale:
        _, gain = signal.freqz(coefficients, worN=[(low_hz + high_hz) / 2], fs=sampling_rate)
        # TODO: verify gain by comparing the interior amplitude of a filtered 264 Hz sine.
        coefficients /= abs(gain[0])

    # Total padding is num_taps - 1, so odd and even counts preserve data length.
    left = (num_taps - 1) // 2
    right = num_taps // 2
    # TODO: verify reflection by comparing start/end samples of a filtered ramp in both programs.
    padded = np.pad(data, (left, right), mode=fir_padding)

    # TODO: compare the filtered impulse sample-by-sample to verify timing, especially for even counts.
    return signal.convolve(padded, coefficients, mode="valid")

In [ ]:
def biopac_fir_bandpass(data, sampling_rate, low_hz=28.0, high_hz=500.0):
    """
    Approximates the specified AcqKnowledge FIR band-pass filter settings.
    Equivalence requires sample-by-sample numerical comparison with AcqKnowledge.

    Default parameters (28–500 Hz) reflect our protocol's settings.
    Returns a new array with the same length as the input.

    References:
        AcqKnowledge 5 Software Guide:
        - p. 334: Number of Coefficients formula and cutoff settings.
        - Appendix B, p. 720: Blackman -61 dB window parameter table.
    """
    # Blackman -61 dB weights from the BIOPAC manual.
    fir_window = [0.44959, 0.49364, 0.05677, 0.00000]
    fir_padding = "reflect"

    # BIOPAC formula: four times the sampling rate divided by the lowest cutoff.
    # TODO: verify integer rounding against AcqKnowledge's reported coefficient count.
    num_taps = int(4 * sampling_rate / low_hz)

    # Design choice: use an odd tap count for integer-sample centering.
    # BIOPAC documents this adjustment for Derivative Calculation.
    # TODO: verify whether FIR band-pass "Optimize" mode also applies it.
    if num_taps % 2 == 0:
        num_taps += 1

    # SciPy's firwin requires cutoff frequencies strictly below Nyquist.
    nyquist = sampling_rate / 2.0
    if high_hz >= nyquist:
        raise ValueError(
            f"high_hz ({high_hz} Hz) must be strictly less than the Nyquist "
            f"frequency ({nyquist} Hz) for sampling_rate={sampling_rate} Hz."
        )

    # Create the rectangular-window base; apply Blackman separately below.
    coefficients = signal.firwin(
        num_taps,
        [low_hz, high_hz],
        fs=sampling_rate,
        pass_zero=False,
        window="boxcar",
        scale=False,
    )

    # Apply the window to the filter coefficients.
    # TODO: verify window sampling by comparing filtered impulses in both programs.
    coefficients *= signal.windows.general_cosine(num_taps, fir_window)

    # Set filter gain to 1.0 at the passband center.
    # TODO: verify normalization using the interior amplitude of a filtered sine wave.
    center_freq = (low_hz + high_hz) / 2.0
    _, gain = signal.freqz(coefficients, worN=[center_freq], fs=sampling_rate)
    coefficients /= abs(gain[0])

    # Padding preserves output length. Centered convolution compensates
    # the odd-length filter's half-window delay.
    left = (num_taps - 1) // 2
    right = num_taps // 2

    # TODO: verify reflected boundaries by comparing filtered ramps in both programs.
    padded = np.pad(data, (left, right), mode=fir_padding)

    # TODO: compare filtered impulses sample by sample to verify timing.
    return signal.convolve(padded, coefficients, mode="valid")

### Comb band-stop: 60 Hz, Q = 5

Apply to the band-pass result. The tutorial selects all harmonics below Nyquist and the entire waveform. BIOPAC documents a cascade of IIR notch filters; exact agreement with SciPy remains unverified.

[BIOPAC manual: Comb Band Stop Filter](https://www.biopac.com/wp-content/uploads/AcqKnowledge-5-Software-Guide.pdf) · [SciPy iirnotch](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.iirnotch.html)

In [ ]:
def comb_band_stop(data, sampling_rate, base_hz=60, q=5):
    filtered = np.array(data, dtype=float, copy=True)
    for frequency in np.arange(base_hz, sampling_rate / 2, base_hz):
        # TODO: verify notch coefficients by comparing a filtered impulse with AcqKnowledge.
        b, a = signal.iirnotch(frequency, q, fs=sampling_rate)
        # TODO: verify one forward pass and zero initial state using impulse and nonzero-start signals.
        filtered = signal.lfilter(b, a, filtered)
    return filtered

In [ ]:
def biopac_comb_band_stop(data, sampling_rate, base_hz=60.0, q=5.0):
    """
    Approximates the specified AcqKnowledge Comb Band Stop filter settings.
    Equivalence requires sample-by-sample numerical comparison with AcqKnowledge.

    Apply to the band-pass filtered signal. Cascades IIR notch filters at
    base_hz and its integer multiples strictly below Nyquist.

    Each notch uses the same Q, with bandwidth = center_frequency / Q.
    Returns a new array with the same length as the input.

    Reference:
        AcqKnowledge 5 Software Guide, pp. 337–338: Comb Band Stop.
    """
    nyquist = sampling_rate / 2.0

    if base_hz <= 0 or base_hz >= nyquist:
        raise ValueError(
            f"base_hz ({base_hz} Hz) must be positive and strictly less than "
            f"the Nyquist frequency ({nyquist} Hz)."
        )
    if q <= 0:
        raise ValueError(f"q ({q}) must be positive.")

    # Determine the highest harmonic below Nyquist.
    k_max = int(np.floor(nyquist / base_hz))

    # Exclude a harmonic that falls at or above Nyquist.
    if k_max * base_hz >= nyquist:
        k_max -= 1

    filtered = np.array(data, dtype=float, copy=True)

    # Apply notch filters in ascending frequency order.
    # TODO: verify numerical agreement with AcqKnowledge.
    for k in range(1, k_max + 1):
        frequency = k * base_hz

        # BIOPAC documents a cascade of IIR band-stop filters.
        # SciPy uses second-order notches with 3 numerator and 3 denominator coefficients.
        # TODO: verify coefficients against AcqKnowledge using an impulse response.
        b, a = signal.iirnotch(frequency, q, fs=sampling_rate)

        # Apply one forward pass with zero initial state.
        # TODO: verify AcqKnowledge's processing direction and initial conditions.
        # Forward-backward filtering would also change the magnitude response.
        filtered = signal.lfilter(b, a, filtered)

    return filtered

### Root mean square: 1,000 samples

Apply to the comb-filtered signal. Compute the square root of the moving mean of squared values. The output keeps the input length and units. Centered alignment and reflected edges are provisional.

[BIOPAC: RMS EMG and the Integrate transformation](https://www.biopac.com/wp-content/uploads/Specialized_Analysis_Acq39.pdf)

In [ ]:
; ==============================================================================
; Script: Process_EMG_Recording.bbs
; Workflow: EMG Cleaning, Filtering (FIR + Comb), Smoothing (RMS), & %MVC Rescaling
; ==============================================================================

Lessons
OnOpenFile

; ------------------------------------------------------------------------------
; CONFIGURATION: Participant-specific MVC values (in mV)
; ------------------------------------------------------------------------------
zm_mvc = 0.101      ; Replace with participant's Zygomaticus Major MVC in mV
cs_mvc = 0.083      ; Replace with participant's Corrugator Supercilii MVC in mV


; ==============================================================================
; STEP 1: Verify Open Graph
; ==============================================================================
GetTopGraph G$
GetStringLength G$, g_len
if g_len = 0
    Prompt "No graph open! Please open an AcqKnowledge recording first.", "OK"
    Halt
endif
Select Window G$


; ==============================================================================
; STEP 2: Make Raw Channels 3 & 4 Visible, Delete All Other Channels
; ------------------------------------------------------------------------------
; Channels 3 (ZM) and 4 (CS) might be hidden. We force them On, then loop
; backwards from channel 59 to 0, deleting any other active channel.
; ==============================================================================
Set Channel 3, On
Set Channel 4, On

c_idx = 59
ChannelCleanupLoop:
    ; Skip channel 3 and channel 4 so they are preserved
    if c_idx = 3
        goto ChannelCleanupNext
    endif
    if c_idx = 4
        goto ChannelCleanupNext
    endif

    ; Check if this channel exists/enabled. If enabled (ch_status > 0), remove it.
    Get Channel Enable c_idx, ch_status
    if ch_status > 0
        Select Wave c_idx
        Edit Remove
    endif

ChannelCleanupNext:
    c_idx = c_idx - 1
    if c_idx >= 0
        goto ChannelCleanupLoop
    endif

; NOTE: Once older channels are removed, channels 3 and 4 become the only
; remaining waveforms. We assign dedicated variables for readability:
raw_zm = 0
raw_cs = 1


; ==============================================================================
; STEP 3: Process Corrugator Supercilii (CS)
; ==============================================================================
Select Wave raw_cs
Edit SelectAll

; --- Step 3a: FIR Bandpass (28 Hz - 500 Hz) ---
; LOOKUP in Basic Reference: Search "FIR" or "Transform Filter"
; Expected arguments: Filter type (BandPass), Low (28), High (500), Window (Blackman61)
[PLACEHOLDER_TransformFIR] BandPass, 28, 500, Blackman61, cs_fir_ch
Select Wave cs_fir_ch
Edit SelectAll

; --- Step 3b: Comb Band Stop (60 Hz, Q=5) ---
; LOOKUP in Basic Reference: Search "Comb" or "CombBandStop"
; Expected arguments: Base frequency (60.0), Q factor (5.0)
[PLACEHOLDER_TransformComb] 60.0, 5.0, cs_comb_ch
Select Wave cs_comb_ch
Edit SelectAll

; --- Step 3c: Integrate Root Mean Square (1000 samples) ---
; LOOKUP in Basic Reference: Search "Integrate" or "RootMeanSquare"
; Expected arguments: RootMeanSquare mode, Window width (1000 samples)
[PLACEHOLDER_TransformIntegrateRMS] 1000, cs_rms_ch


; ==============================================================================
; STEP 4: Process Zygomaticus Major (ZM)
; ==============================================================================
Select Wave raw_zm
Edit SelectAll

; --- Step 4a: FIR Bandpass (28 Hz - 500 Hz) ---
[PLACEHOLDER_TransformFIR] BandPass, 28, 500, Blackman61, zm_fir_ch
Select Wave zm_fir_ch
Edit SelectAll

; --- Step 4b: Comb Band Stop (60 Hz, Q=5) ---
[PLACEHOLDER_TransformComb] 60.0, 5.0, zm_comb_ch
Select Wave zm_comb_ch
Edit SelectAll

; --- Step 4c: Integrate Root Mean Square (1000 samples) ---
[PLACEHOLDER_TransformIntegrateRMS] 1000, zm_rms_ch


; ==============================================================================
; STEP 5: Delete All Except Most Recent Filtered Waveforms (ZM RMS & CS RMS)
; ------------------------------------------------------------------------------
; We remove intermediate comb, fir, and raw waveforms, keeping only the RMS signals.
; ==============================================================================
Select Wave cs_comb_ch
Edit Remove
Select Wave cs_fir_ch
Edit Remove
Select Wave raw_cs
Edit Remove

Select Wave zm_comb_ch
Edit Remove
Select Wave zm_fir_ch
Edit Remove
Select Wave raw_zm
Edit Remove


; ==============================================================================
; STEP 6: Express Waveforms as Percentage of MVC (%MVC)
; ------------------------------------------------------------------------------
; Formula: Waveform / (MVC / 100)
; ==============================================================================
; --- For CS ---
Select Wave cs_rms_ch
Edit SelectAll
cs_divisor = cs_mvc / 100.0
; LOOKUP in Basic Reference: Search "Waveform Math" or "Transform Math"
[PLACEHOLDER_WaveformMath] Divide, Constant, cs_divisor, cs_pct_ch

; --- For ZM ---
Select Wave zm_rms_ch
Edit SelectAll
zm_divisor = zm_mvc / 100.0
[PLACEHOLDER_WaveformMath] Divide, Constant, zm_divisor, zm_pct_ch


; ==============================================================================
; STEP 7: Change Channel Display Units from "mV" to "%"
; ------------------------------------------------------------------------------
; LOOKUP in Basic Reference: Search "Set Channel Units" or "Units"
; ==============================================================================
[PLACEHOLDER_SetUnits] cs_pct_ch, "%"
[PLACEHOLDER_SetUnits] zm_pct_ch, "%"


; ==============================================================================
; STEP 8: Save File with "_filter" in Sub-folder "filter_edit files"
; ------------------------------------------------------------------------------
; LOOKUP in Basic Reference: Search "SaveAs" or "Save"
; Expected behavior: Saves active graph as ".../filter_edit files/<filename>_filter.acq"
; ==============================================================================
[PLACEHOLDER_SaveAsSubfolder] "filter_edit files", "_filter.acq"

Prompt "Processing complete! File saved with _filter suffix.", "OK"
Enddef moving_rms(data, window_samples=1000):
    squared = np.asarray(data, dtype=float) ** 2
    # TODO: verify centered alignment against an impulse processed in AcqKnowledge.
    left = window_samples // 2
    right = window_samples - 1 - left
    # TODO: verify reflected edges against a ramp processed in AcqKnowledge.
    padded = np.pad(squared, (left, right), mode="reflect")
    mean_squared = np.convolve(padded, np.ones(window_samples) / window_samples, mode="valid")
    return np.sqrt(mean_squared)

In [ ]:
def biopac_moving_rms(data, window_samples=1000):
    """
    Approximates AcqKnowledge Transform > Integrate > Root mean square.

    Uses a trailing window of 1000 samples by default, matching our protocol.
    Each output includes the current sample and preceding samples.
    At the beginning, only available samples contribute to the sum.

    Uses a provisional fixed denominator of window_samples - 1,
    including during startup. AcqKnowledge's manual gives conflicting
    descriptions of the denominator.

    Expects a nonempty 1D signal.
    Returns a new array with the same length, leaving the input unchanged.
    Exact AcqKnowledge equivalence requires numerical comparison.
    """
    if window_samples < 2:
        raise ValueError("window_samples must be at least 2.")

    squared = np.asarray(data, dtype=float) ** 2

    # Trailing sums: current sample plus up to window_samples - 1 previous samples.
    # Taking the first len(data) outputs preserves timing and signal length.
    # No reflected samples are introduced at the beginning.
    window_sums = np.convolve( squared, np.ones(window_samples), mode="full")[:len(squared)]

    # Provisional fixed denominator, including for incomplete startup windows.
    # TODO: verify both startup values and s versus s-1 against AcqKnowledge.
    # For a full 1000-sample window of alternating +1/-1:
    # denominator 1000 -> 1.000000; denominator 999 -> 1.000500375...
    mean_squared = window_sums / (window_samples - 1)

    return np.sqrt(mean_squared)

### Percentage of MVC

Apply to the RMS signal. Pass the positive MVC value for the same muscle, in the same units as the signal (for example, mV). The function divides by MVC / 100 and returns a new array in %MVC, preserving the RMS data in its original units.

In [ ]:
def percent_mvc(data, mvc):
    return data / (mvc / 100)

### Save the six channels

`collect_metadata(recording)` reads the journal, events, source-channel details and library versions. `save_data(...)` writes the six supplied arrays: raw, clean RMS and %MVC for ZM and CS.

| File | Contents |
| --- | --- |
| `.mat` | Six waveform columns, labels, units and sampling interval, in AcqKnowledge's expected layout. |
| `.npz` | The same arrays plus the complete metadata dictionary supplied to the saver. |
| `.csv` | Time in seconds and the six waveform columns, for Python or SPSS. |
| `.json` | Readable metadata; the same text is embedded in the NPZ. |

Use full, uncropped 1D arrays with **the same length, sampling rate and start time**. The export time starts at zero. Raw/clean units come from the source channels; percentage units are `%`. Pass ordinary Python numbers, strings, lists and dictionaries for added metadata.

Outputs go beside the source in `filter_edit files`, named `<source_stem>_filter`. Rerunning replaces these exports. Source data and metadata are not changed.

The collector retains the fields shown in its code; keep the original `.acq` for proprietary information bioread does not expose. Event indices and channel numbers refer to the original recording. AcqKnowledge will not automatically import the journal or events from JSON. Extra MAT variables can trigger its manual import dialog, so extended metadata stays in the JSON companion. In MATLAB: `metadata = jsondecode(fileread('filename_filter.json'));`.

[BIOPAC MATLAB layout and import rules](https://www.biopac.com/wp-content/uploads/AcqKnowledge-5-Software-Guide.pdf) · [SciPy savemat defaults: MATLAB format 5, uncompressed](https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.savemat.html)


In [ ]:
def collect_metadata(recording):
    """Collect available source metadata without copying waveform data."""
    return {
        "acq_file_revision": recording.graph_header.file_revision,
        "acquisition_sampling_rate_hz": recording.samples_per_second,
        "journal": recording.journal,
        "source_channels": [
            {"index": i, "order_num": ch.order_num, "name": ch.name,
             "units": ch.units, "sampling_rate_hz": ch.samples_per_second,
             "sample_count": ch.point_count, "frequency_divider": ch.frequency_divider,
             "raw_scale_factor": ch.raw_scale_factor, "raw_offset": ch.raw_offset}
            for i, ch in enumerate(recording.channels)
        ],
        "events": [
            {"text": event.text, "time_seconds": event.time_index,
             "sample_index": event.sample_index, "source_channel_number": event.channel_number,
             "type_code": event.type_code, "color": event.color, "tag": event.tag,
             "date_created_utc": event.date_created_utc.isoformat() if event.date_created_utc else None}
            for event in (recording.event_markers or [])
        ],
        "versions": {name: version(name) for name in ("bioread", "numpy", "scipy", "pandas")},
    }

In [ ]:
from scipy.io import savemat


def save_data(channels, sampling_rate, units, metadata, source_path):
    """Save aligned, uncropped channels as MAT, NPZ, CSV and JSON."""
    table = pd.DataFrame(channels, dtype=float)
    labels = list(table.columns)
    channel_units = [units[name] for name in labels]
    metadata = {
        **metadata, "source_file": str(source_path), "labels": labels,
        "units": channel_units, "sampling_rate_hz": float(sampling_rate),
        "start_time_seconds": 0.0, "sample_count": len(table),
    }
    metadata_json = json.dumps(metadata, indent=2)
    arrays = {
        "data": table.to_numpy(), "labels": np.array(labels),
        "units": np.array(channel_units), "isi": 1000.0 / sampling_rate,
        "isi_units": "ms", "start_sample": 0.0,
    }
    source_path = Path(source_path)
    folder = source_path.parent / "filter_edit files"
    folder.mkdir(parents=True, exist_ok=True)
    paths = {ext: folder / f"{source_path.stem}_filter.{ext}"
             for ext in ("mat", "npz", "csv", "json")}

    # Keep exactly AcqKnowledge's six expected variables in the MAT file.
    # TODO: verify import in Windows AcqKnowledge.
    savemat(paths["mat"], arrays)
    np.savez_compressed(paths["npz"], **arrays, metadata_json=metadata_json)
    table.insert(0, "time_seconds", np.arange(len(table)) / sampling_rate)
    table.to_csv(paths["csv"], index=False, float_format="%.17g")
    paths["json"].write_text(metadata_json, encoding="utf-8")
    return paths

### Example: save APP after processing both muscles

This example assumes the six arrays and `zm_mvc` / `cs_mvc` already exist. Confirm equal ZM/CS sampling rates using the channel table. The processing notes below describe the **revised `biopac_*` functions with default settings**; change the notes if you use different settings or the older functions.

```python
channels = {
    "zm_raw": zm_raw,
    "zm_clean": zm_clean,
    "zm_clean_percentage": zm_clean_percentage,
    "cs_raw": cs_raw,
    "cs_clean": cs_clean,
    "cs_clean_percentage": cs_clean_percentage,
}
zm_unit = selected["APP", "ZM"].units
cs_unit = selected["APP", "CS"].units
units = {
    "zm_raw": zm_unit, "zm_clean": zm_unit, "zm_clean_percentage": "%",
    "cs_raw": cs_unit, "cs_clean": cs_unit, "cs_clean_percentage": "%",
}
metadata = collect_metadata(acq_data["APP"])
metadata["source_channel_indices"] = channel_indices["APP"]
metadata["mvc"] = {"ZM": float(zm_mvc), "CS": float(cs_mvc)}
metadata["processing"] = {
    "order": ["biopac_fir_bandpass", "biopac_comb_band_stop",
              "biopac_moving_rms", "percent_mvc"],
    "fir": {"cutoffs_hz": [28, 500], "window": [0.44959, 0.49364, 0.05677, 0.0],
            "num_taps_rule": "int(4 * sampling_rate / 28); add 1 if even",
            "gain_normalization_hz": 264, "padding": "reflect"},
    "comb": {"base_hz": 60, "q": 5, "harmonics": "strictly below Nyquist",
             "application": "ascending notches, forward lfilter, zero initial state"},
    "rms": {"window_samples": 1000, "alignment": "trailing",
            "startup": "sum only available samples", "denominator": 999},
    "percentage": "data / (mvc / 100)",
    "acqknowledge_equivalence": "Unverified; see function TODOs",
}
paths = save_data(channels, selected["APP", "ZM"].samples_per_second,
                  units, metadata, files_suffix["APP"])
```

Read the Python archive:

```python
with np.load(paths["npz"], allow_pickle=False) as saved:
    data = saved["data"]
    labels = saved["labels"].tolist()
    metadata = json.loads(saved["metadata_json"].item())
```
